# SA_E1A — Stage 3: Rewrite Classification

This notebook classifies each participant's open-ended rewrite as **categorical**, **conditional**, or **ambiguous** — blind to their forced-choice interpretation response.

**What this does, in plain terms:**

Each participant saw a sentence like *"A community garden plot, being necessary for growing vegetables, the right of people to plant and cultivate their own produce, must not be infringed."* and was asked to rewrite it in their own words. Separately, they chose a forced-choice interpretation (categorical vs. conditional).

For H5a/H5b/H5c, we need to know whether each person's *rewrite* reflects a categorical or conditional reading — independent of what they chose on the forced-choice question. This is a semantic judgment task: does the rewrite present the right as always holding (categorical), or as depending on whether the condition in the prefatory clause is met (conditional)?

We send each rewrite to Claude along with the original sentence, and ask it to classify the rewrite. The model does **not** see the participant's forced-choice answer.

**Output:** A CSV file (`SA_E1A_rewrite_classifications.csv`) with columns: `prolific_pid`, `sentence_id`, `rewrite_code`, `justification`.

## 1. Setup

In [1]:
# ──────────────────────────────────────────────────────────────────────
# INSTALL ANTHROPIC SDK
# ──────────────────────────────────────────────────────────────────────

!pip install anthropic -q
!pip install --upgrade anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.5/627.5 kB 24.2 MB/s eta 0:00:00


In [2]:
import anthropic
import csv
import json
import time
import os
from google.colab import files, userdata

# ──────────────────────────────────────────────────────────────────────
# API KEY
# Store your Anthropic API key in Colab's Secrets (key icon in left
# sidebar) with the name ANTHROPIC_API_KEY. This avoids pasting the
# key directly in the notebook.
#
# Alternatively, uncomment the line below and paste your key directly
# (but don't share the notebook with the key in it).
# ──────────────────────────────────────────────────────────────────────

try:
    api_key = userdata.get('ANTHROPIC_API_KEY')
except:
    api_key = input('Enter your Anthropic API key: ')

api_key = userdata.get('ANTHROPIC_API_KEY').strip()

client = anthropic.Anthropic(api_key=api_key)
print('Anthropic client initialized.')

Anthropic client initialized.


In [3]:
# ──────────────────────────────────────────────────────────────────────
# LOAD DATA
# Upload SA_E1A_analytic.csv (the output of Stage 2) when prompted.
# ──────────────────────────────────────────────────────────────────────

uploaded = files.upload()  # Upload SA_E1A_analytic.csv

filename = list(uploaded.keys())[0]
with open(filename) as f:
    reader = csv.DictReader(f)
    participants = list(reader)

print(f'Loaded {len(participants)} participants.')
print(f'Sample rewrite: {participants[0]["rewrite"][:100]}...')

Saving SA_E1A_analytic.csv to SA_E1A_analytic.csv
Loaded 100 participants.
Sample rewrite: it is important for a community to be as one in order to respect its tradition. You must not disrega...


## 2. Classification Prompt

The prompt is the most important part of this notebook. It defines what the classifier does.

**Key design decisions:**

- The model sees the **original sentence** and the **rewrite only**. It does not see the forced-choice answer, confidence rating, or any other response. This preserves the "blind coding" requirement.
- The model classifies into three categories: **categorical** (the right is presented as unconditional), **conditional** (the right is presented as depending on the necessity condition), or **ambiguous** (cannot clearly determine).
- The model provides a brief justification for each classification, which allows us to audit its reasoning and catch errors.
- We explicitly instruct the model about the distinction between *causal explanation* ("because X, therefore Y" — categorical) and *contingency* ("only when X is true" — conditional), since this is the subtlest judgment call.

In [4]:
# ──────────────────────────────────────────────────────────────────────
# CLASSIFICATION PROMPT
#
# This is the system prompt that defines the classification task.
# It is sent with every API call. The user message will contain the
# specific original sentence and participant rewrite.
# ──────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a linguistic analyst coding open-ended rewrite responses in a psycholinguistics study.

Participants read a sentence with the structure:
  "A [subject], being necessary for [purpose], the right of [people] to [action], must not be [restricted]."

They were then asked to rewrite the sentence in their own words. Your task: Based ONLY on the rewrite, classify whether the participant expressed a CATEGORICAL or CONDITIONAL reading.

CATEGORICAL means: The rewrite presents the right as holding unconditionally. The right is stated as a fact, an entitlement, or a principle that simply exists — it is not scoped or limited to situations where the necessity condition is met.

ALL of the following patterns are CATEGORICAL:
- The right is asserted as always holding: "People have the right to trade goods."
- Causal/justificatory language connecting necessity to the right: "Because kitchens are needed for cooking, people must be allowed to set up cooking spaces." (The necessity explains WHY the right exists, not WHEN it applies.)
- "Since," "therefore," "so," or "as" linking necessity to the right: "Since a field is necessary for harvest, the right to farm must not be infringed."
- Dropping the prefatory clause entirely: "The postal service is a right and must not be disrupted." (If the participant didn't include the necessity condition, they are treating the right as independent of it.)
- Dropping the operative clause and only restating the necessity: "A functional kitchen is necessary for meal preparation." (The participant focused on the factual claim without introducing any conditionality.)
- Combining the prefatory and operative clauses with "and": "A working bus system and the right to public transportation must not be revoked." (Joining them as co-equal statements, not making one depend on the other.)
- Any rewrite that asserts the right WITHOUT language restricting its scope to specific conditions.

CONDITIONAL means: The rewrite explicitly presents the right as CONTINGENT on the necessity condition being met. The right applies only when, only if, or only in cases where the subject is necessary. You must find explicit scope-limiting language.

Examples of CONDITIONAL rewrites:
- "People have the right to trade goods ONLY WHEN a marketplace is needed." (scope-limited)
- "IF weather stations are necessary for predicting storms, THEN people must be allowed to respond." (if-then contingency)
- "IN CASES WHERE a food reserve is needed, the right to store provisions is protected." (restricted scope)
- "A free press THAT IS USED TO inform the public has the right to publish information." (the relative clause restricts which press institutions have the right — only those actually serving the informational function)

AMBIGUOUS means: The rewrite is genuinely uninterpretable. Reserve this category ONLY for:
- Responses that are so garbled or grammatically broken that you truly cannot determine what the participant meant.
- Responses that are incomplete fragments (e.g., trailing off mid-sentence with no main clause).
- Responses that are near-verbatim copies of the original sentence with only trivial changes (e.g., replacing "being necessary" with "is necessary" and nothing else), since the participant did not transform the sentence enough to reveal their interpretation.

IMPORTANT: Do NOT classify a rewrite as ambiguous simply because the logical relationship between the clauses is not spelled out explicitly. If the rewrite asserts the right without introducing conditional or scope-limiting language, classify it as CATEGORICAL. The default classification is CATEGORICAL — only classify as CONDITIONAL if you find explicit contingency language, and only classify as AMBIGUOUS if the response is genuinely uninterpretable or a near-verbatim copy.

Respond with EXACTLY this JSON format and nothing else:
{"classification": "categorical" or "conditional" or "ambiguous", "justification": "One sentence explaining your reasoning."}"""

print('System prompt defined.')
print(f'Length: {len(SYSTEM_PROMPT)} characters.')

System prompt defined.
Length: 3984 characters.


In [5]:
# ──────────────────────────────────────────────────────────────────────
# CLASSIFICATION FUNCTION
#
# Sends one rewrite to the API and returns the classification.
# Includes retry logic for transient API errors and rate limiting.
# ──────────────────────────────────────────────────────────────────────

def classify_rewrite(original_sentence, rewrite, max_retries=3):
    """
    Classify a single rewrite as categorical, conditional, or ambiguous.

    Args:
        original_sentence: The full original sentence the participant saw.
        rewrite: The participant's open-ended rewrite.
        max_retries: Number of times to retry on API errors.

    Returns:
        dict with keys 'classification' and 'justification', or
        dict with 'classification': 'error' and 'justification': error message.
    """

    # Build the user message. The model sees ONLY the original sentence
    # and the rewrite — no forced-choice answer, no confidence, nothing else.
    user_message = (
        f"Original sentence:\n{original_sentence}\n\n"
        f"Participant's rewrite:\n{rewrite}\n\n"
        f"Classify this rewrite."
    )

    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model='claude-sonnet-4-20250514',
                max_tokens=200,
                system=SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': user_message}]
            )

            # Extract the text response
            text = response.content[0].text.strip()

            # Parse the JSON
            # Sometimes the model wraps in markdown code fences; strip those
            text = text.replace('```json', '').replace('```', '').strip()
            result = json.loads(text)

            # Validate
            if result.get('classification') not in ('categorical', 'conditional', 'ambiguous'):
                raise ValueError(f"Invalid classification: {result.get('classification')}")

            return result

        except anthropic.RateLimitError:
            # Rate limited — wait and retry
            wait = 2 ** (attempt + 1)
            print(f'  Rate limited, waiting {wait}s...')
            time.sleep(wait)

        except (json.JSONDecodeError, ValueError, KeyError) as e:
            # Parse error — retry with fresh call
            print(f'  Parse error on attempt {attempt+1}: {e}')
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                return {'classification': 'error', 'justification': f'Parse error: {str(e)}. Raw: {text[:200]}'}

        except Exception as e:
            # Other API error
            print(f'  API error on attempt {attempt+1}: {e}')
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return {'classification': 'error', 'justification': str(e)}

    return {'classification': 'error', 'justification': 'Max retries exceeded'}

## 3. Run Classification

We classify all 100 rewrites. This takes approximately 2–3 minutes depending on API latency (each call takes ~1–2 seconds). We add a small delay between calls to avoid rate limits.

In [6]:
# Test: can Colab reach the Anthropic API at all?
import httpx

try:
    r = httpx.get("https://api.anthropic.com", timeout=10)
    print(f"Connection OK (status {r.status_code})")
except Exception as e:
    print(f"Connection failed: {e}")

# Also verify key name matches
from google.colab import userdata
try:
    key = userdata.get('ANTHROPIC_API_KEY')
    print(f"Key retrieved: starts with {key[:10]}...")
except Exception as e:
    print(f"Key retrieval failed: {e}")

Connection OK (status 404)
Key retrieved: starts with sk-ant-api...


In [7]:
# Test a single API call
import anthropic

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

response = client.messages.create(
    model='claude-sonnet-4-20250514',
    max_tokens=50,
    messages=[{'role': 'user', 'content': 'Say hello in one word.'}]
)
print(response.content[0].text)

Hello!


In [8]:
# ──────────────────────────────────────────────────────────────────────
# CLASSIFY ALL REWRITES
#
# We iterate through all participants, send their rewrite to the API,
# and collect the results. Progress is printed every 10 participants.
# ──────────────────────────────────────────────────────────────────────

results = []
errors = []

print(f'Classifying {len(participants)} rewrites...\n')

for i, p in enumerate(participants):
    # Call the classifier
    result = classify_rewrite(
        original_sentence=p['full_sentence'],
        rewrite=p['rewrite']
    )

    # Store result
    results.append({
        'prolific_pid': p['prolific_pid'],
        'sentence_id': p['sentence_id'],
        'rewrite_code': result['classification'],
        'justification': result['justification']
    })

    # Track errors
    if result['classification'] == 'error':
        errors.append((p['prolific_pid'], result['justification']))

    # Progress
    if (i + 1) % 10 == 0 or i == len(participants) - 1:
        print(f'  [{i+1}/{len(participants)}] classified')

    # Small delay to avoid rate limits (Sonnet is generous but let's be safe)
    time.sleep(0.5)

print(f'\nDone. Errors: {len(errors)}')
if errors:
    for pid, err in errors:
        print(f'  PID={pid}: {err}')

Classifying 100 rewrites...

  [10/100] classified
  [20/100] classified
  [30/100] classified
  [40/100] classified
  [50/100] classified
  [60/100] classified
  [70/100] classified
  [80/100] classified
  [90/100] classified
  [100/100] classified

Done. Errors: 0


## 4. Review Results

In [9]:
# ──────────────────────────────────────────────────────────────────────
# CLASSIFICATION DISTRIBUTION
# ──────────────────────────────────────────────────────────────────────

from collections import Counter

codes = Counter(r['rewrite_code'] for r in results)
total = len(results)

print('Classification distribution:')
for code in ['categorical', 'conditional', 'ambiguous', 'error']:
    if code in codes:
        print(f'  {code:15s}: {codes[code]:3d} ({codes[code]/total*100:.1f}%)')

print(f'\n  Total: {total}')

Classification distribution:
  categorical    :  89 (89.0%)
  conditional    :   2 (2.0%)
  ambiguous      :   9 (9.0%)

  Total: 100


In [10]:
# ──────────────────────────────────────────────────────────────────────
# CONCORDANCE PREVIEW
#
# Compare the rewrite classification (blind) against the forced-choice
# interpretation. This is the core data for H5a/H5b/H5c.
#
# NOTE: The classifier was blind to the forced-choice answer. This
# comparison is for OUR review only — it tells us how often the
# rewrite and forced-choice agree.
# ──────────────────────────────────────────────────────────────────────

# Build a lookup from PID to forced-choice interpretation
fc_lookup = {p['prolific_pid']: p['interpretation'] for p in participants}
# Interpretation coding: 1=conditional, 2=categorical, 3=neither

# Map forced-choice to labels
fc_labels = {'1': 'conditional', '2': 'categorical', '3': 'neither'}

print('Concordance: Forced-Choice vs. Rewrite Classification')
print('=' * 60)

# Build cross-tabulation
crosstab = {}  # (forced_choice_label, rewrite_code) -> count
for r in results:
    fc = fc_labels.get(fc_lookup.get(r['prolific_pid'], '?'), '?')
    rc = r['rewrite_code']
    key = (fc, rc)
    crosstab[key] = crosstab.get(key, 0) + 1

# Display as a table
rc_categories = ['categorical', 'conditional', 'ambiguous']
fc_categories = ['categorical', 'conditional', 'neither']

# Header
print(f'{"":>20s}', end='')
for rc in rc_categories:
    print(f'{rc:>15s}', end='')
print(f'{"TOTAL":>10s}')

print('-' * 60)

# Rows
for fc in fc_categories:
    row_total = sum(crosstab.get((fc, rc), 0) for rc in rc_categories)
    print(f'FC {fc:>16s}', end='')
    for rc in rc_categories:
        count = crosstab.get((fc, rc), 0)
        print(f'{count:>15d}', end='')
    print(f'{row_total:>10d}')

print()
print('FC = forced-choice interpretation')
print('Columns = rewrite classification (blind)')

Concordance: Forced-Choice vs. Rewrite Classification
                        categorical    conditional      ambiguous     TOTAL
------------------------------------------------------------
FC      categorical             80              1              5        86
FC      conditional              5              0              4         9
FC          neither              4              1              0         5

FC = forced-choice interpretation
Columns = rewrite classification (blind)


In [11]:
# ──────────────────────────────────────────────────────────────────────
# SPOT-CHECK: Show rewrites where forced-choice and rewrite DISAGREE
#
# These are the interesting cases — participants whose rewrite says
# one thing and whose forced-choice says another. Reviewing these
# helps verify the classifier is making reasonable calls.
# ──────────────────────────────────────────────────────────────────────

print('DISCORDANT CASES (forced-choice ≠ rewrite code)')
print('=' * 80)

for r, p in zip(results, participants):
    fc = fc_labels.get(p['interpretation'], '?')
    rc = r['rewrite_code']

    # Check for discordance (ignoring 'neither' and 'ambiguous' as special)
    is_discordant = (
        (fc == 'categorical' and rc == 'conditional') or
        (fc == 'conditional' and rc == 'categorical')
    )

    if is_discordant:
        print(f'\nPID: {p["prolific_pid"]}')
        print(f'  Sentence {p["sentence_id"]}: {p["full_sentence"][:80]}...')
        print(f'  Forced-choice: {fc}')
        print(f'  Rewrite code:  {rc}')
        print(f'  Rewrite:       {p["rewrite"][:120]}')
        print(f'  Justification: {r["justification"]}')

DISCORDANT CASES (forced-choice ≠ rewrite code)

PID: 5b9845dfae52a00001482c02
  Sentence 2: A respected cultural tradition, being necessary for fostering unity within a com...
  Forced-choice: conditional
  Rewrite code:  categorical
  Rewrite:       A tradition that is well respected that keeps people united in a community, and the right of freedom of information shou
  Justification: The participant combines the necessity statement and the right assertion with 'and', treating them as co-equal statements rather than making the right contingent on the necessity condition, and asserts the right without any scope-limiting language.

PID: 6632ed07883d9dc97c9c7ac7
  Sentence 6: A structurally sound dam, being necessary for protection from rising waters, the...
  Forced-choice: conditional
  Rewrite code:  categorical
  Rewrite:       As a dam to protect from rising water is like the right of people to defend their land must not be taken away.
  Justification: The rewrite asserts that the 

## 5. Save Results

In [12]:
# ──────────────────────────────────────────────────────────────────────
# SAVE CLASSIFICATIONS
#
# Write the results to a CSV that will be read into the R Markdown
# for Stage 4 (hypothesis testing).
#
# Columns:
#   prolific_pid   — participant identifier (for joining)
#   sentence_id    — which sentence they saw
#   rewrite_code   — categorical, conditional, or ambiguous
#   justification  — the model's reasoning (for auditing)
# ──────────────────────────────────────────────────────────────────────

output_filename = 'SA_E1A_rewrite_classifications.csv'

with open(output_filename, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['prolific_pid', 'sentence_id',
                                           'rewrite_code', 'justification'])
    writer.writeheader()
    writer.writerows(results)

print(f'Saved {len(results)} classifications to {output_filename}')

# Download the file
files.download(output_filename)
print('Download started.')

Saved 100 classifications to SA_E1A_rewrite_classifications.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.


## 6. Summary

**What to do next:**

1. Review the concordance table and the discordant cases above. If any classifications look wrong, you can re-examine the rewrite and justification.
2. Place `SA_E1A_rewrite_classifications.csv` in the same data directory as the other Stage outputs.
3. Proceed to Stage 4 (Hypothesis Testing in R), which will read this file and use the `rewrite_code` column for H5a/H5b/H5c.